In [83]:
import numpy as np
import random 
import pandas as pd
import itertools

In [2]:
random.seed(1)

number_of_decks = 2
sims = 1000000
deck_pen = 0.5

In [ ]:
cards = ['A','2','3','4','5','6','7','8','9','10','J','Q','K']

card_values = {
    #'A' : [1, 11],
    'A' : [11],
    '2' : [2],
    '3' : [3],
    '4' : [4],
    '5' : [5],
    '6' : [6],
    '7' : [7],
    '8' : [8],
    '9' : [9],
    '10' : [10],
    'J' : [10],
    'Q' : [10],
    'K' : [10]
}

decks = []
for i in range(number_of_decks):
    for j in range(len(cards)):
        input = cards[j]
        decks.append(input)
        decks.append(input)
        decks.append(input)
        decks.append(input)

random.shuffle(decks)


In [16]:
card_values['A']

[11]

In [95]:
dealer_face = cards.copy() 

dealt_cards = []
for i in range(len(cards)):
    for j in range(len(cards)):
        if j < i:
            continue
        else:
            dealt_cards.append([cards[i],cards[j]])

results = []
for i in dealer_face:
    for j in dealt_cards:
        results.append({'dealer_face': ','.join(sorted(i)), 'dealt_cards': ','.join(sorted(j))})

results_df = pd.DataFrame(results)

Moves:
first move only:
    double down - double the bet and take one more card
    split - seperate tow cards of the same calue into two hands
    surrender - forfeit the hand for half the bet back
    insurance - dealer face up card is ace but one-halg oringal bet that dealer has blackjack, pays 2 to 1
hit - take another cards
stand - take no more cards

Dealer 
hit until 17
hit soft 17 is a variation

In [96]:
results_df


,dealer_face,dealt_cards
0,A,"A,A"
1,A,"2,A"
2,A,"3,A"
3,A,"4,A"
4,A,"5,A"
...,...,...
1178,K,"J,Q"
1179,K,"J,K"
1180,K,"Q,Q"
1181,K,"K,Q"


In [97]:
def game_sim(number_of_decks = 2,
             sims = 100,
             deck_pen = 0.5):
    
    def deal_cards(current_card):
        player_cards = [decks[current_card], decks[current_card + 2]]
        dealer_cards = [decks[current_card + 1], decks[current_card + 3]]
        current_card = current_card + 4
        return player_cards, dealer_cards
    
    def next_move(first_move, current_card, card_list):
        if first_move == 'Y':
            #add in other moves here
            return random.choice([hit(current_card, card_list), stand(card_list)])
        else:
            return random.choice([hit(current_card, card_list), stand(card_list)])
    
    def hit(current_card, card_list):
        first_move = 'N'
        card_list.append(decks[current_card])
        current_card += 1
        if hand_values(card_list) > 21:
            return 'Bust'
        else:
            next_move(first_move, current_card, card_list)
    
    def stand(card_list):
        return hand_values(card_list)
    
    def dealer_move(dealer_hand, current_card):
        #include soft 17 here
        if hand_values(dealer_hand) > 21:
            return 'Bust'
        if hand_values(dealer_hand) < 17:
            dealer_move(dealer_hand.append(decks[current_card]), current_card + 1)
        else:
            return stand(dealer_hand)
         
    # this needs to deal with aces
    def hand_values(card_list):
        output = 0
        for i in card_list:
            output += card_values[i][0]
        return output
    
    current_card = 0
    while current_card < round(len(decks)*deck_pen):
        first_move = 'Y'
        player_cards, dealer_cards = deal_cards(current_card)
        player_result = next_move(first_move, current_card, player_cards)

        if player_result == 'Bust':
            print(player_cards[:2])
            results_df.loc[(results_df[dealer_face] == dealer_cards[0]) & (results_df[player_cards] == ','.join(sorted(player_cards[:2]))), 'player_move_loss' ] += 1
            results_df.loc[(results_df[dealer_face] == dealer_cards[0]) & (results_df[player_cards] == ','.join(sorted(player_cards[:2]))), 'player_move_total' ] += 1
        else:
            dealer_result = dealer_move(dealer_cards, current_card)
            if dealer_result == 'Bust':
                results_df.loc[(results_df[dealer_face] == dealer_cards[0]) & (results_df[player_cards] == ','.join(sorted(player_cards[:2]))), 'player_move_won' ] += 1
                results_df.loc[(results_df[dealer_face] == dealer_cards[0]) & (results_df[player_cards] == ','.join(sorted(player_cards[:2]))), 'player_move_total' ] += 1
            elif dealer_result > player_result:
                results_df.loc[(results_df[dealer_face] == dealer_cards[0]) & (results_df[player_cards] == ','.join(sorted(player_cards[:2]))), 'player_move_loss' ] += 1
                results_df.loc[(results_df[dealer_face] == dealer_cards[0]) & (results_df[player_cards] == ','.join(sorted(player_cards[:2]))), 'player_move_total' ] += 1
            else:
                print(dealer_cards[0])
                print(','.join(sorted(player_cards[:2])))
                results_df.loc[(results_df[dealer_face] == dealer_cards[0]) & (results_df[player_cards] == ','.join(sorted(player_cards[:2]))), 'player_move_won' ] += 1
                results_df.loc[(results_df[dealer_face] == dealer_cards[0]) & (results_df[player_cards] == ','.join(sorted(player_cards[:2]))), 'player_move_total' ] += 1


In [98]:
game_sim()

['A', '3']


KeyError: "None of [Index(['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K'], dtype='object')] are in the [columns]"